# Полный анализ датасета тестирования детей по истории

**Цель:** выявить аномалии двумя способами:
1. **Ручная проверка правил** — каждое правило в отдельной ячейке. Исправимые ошибки исправляются прямо в датасете; неисправимые — помечаются как аномалии.
2. **Isolation Forest** — обучается на очищенных данных и выявляет статистические выбросы, которые не нарушают ни одного из формальных правил, но при этом сильно отличаются от «типичных» записей.

**Источники правил:**
- `analysis_hakaton.ipynb` (исходный разведочный ноутбук команды)
- `step1_manual_anomalies.py` (10 проверок по заданию ЦИТиС)
- Задание «ЦИТиС 26.docx»

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from datetime import date, datetime
from collections import defaultdict, Counter

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

DATA_FILE = 'hakaton.csv'
OUT_DIR   = 'results_notebook'
os.makedirs(OUT_DIR, exist_ok=True)

TODAY = date(2026, 4, 17)

df = pd.read_csv(DATA_FILE, sep=';', dtype=str, keep_default_na=False)
df = df.fillna('')

print(f'Загружено записей: {len(df)}')
print(f'Колонки: {list(df.columns)}')
df.head(3)

## Вспомогательный инструментарий

Создаём столбцы для трекинга аномалий и правок, а также вспомогательную функцию `flag()` для регистрации нарушений.

In [ ]:
# Столбец для категорий нарушений (может быть несколько через |)
df['anomaly_flags'] = ''
# Столбец для хранения описания правок
df['fixes_applied'] = ''

anomalies = []  # список словарей для CSV-отчёта
fixes_log = []  # список словарей для CSV-лога правок

def flag(idx, category, description):
    """Регистрирует аномалию в строке idx."""
    existing = df.at[idx, 'anomaly_flags']
    df.at[idx, 'anomaly_flags'] = (existing + '|' + category).lstrip('|')
    anomalies.append({
        'row_idx': idx,
        'our_number': df.at[idx, 'our_number'],
        'child': f"{df.at[idx,'last_name']} {df.at[idx,'first_name']}",
        'category': category,
        'description': description,
    })

def fix_log(idx, field, old_val, new_val, reason):
    """Логирует исправление."""
    existing = df.at[idx, 'fixes_applied']
    df.at[idx, 'fixes_applied'] = (existing + '|' + field).lstrip('|')
    fixes_log.append({
        'row_idx': idx,
        'our_number': df.at[idx, 'our_number'],
        'field': field,
        'old_value': old_val,
        'new_value': new_val,
        'reason': reason,
    })

print('Вспомогательные функции готовы.')

---
## Правило 1. Нормализация поля `result` (результат тестирования)

**Проблема:** поле `result` должно принимать ровно два значения — `«Достаточный»` и `«Недостаточный»`. В датасете обнаружены 7 вариантов из-за несогласованного регистра при вводе:

- `«достаточный»`, `«ДОСТАТОЧНЫЙ»` — опечатки регистра
- `«НЕдостаточный»` — смешанный регистр (ошибка ввода)

**Стратегия: FIX (исправление).** Если слово содержит «достаточн» (без учёта регистра) и не содержит «не» — это «Достаточный», иначе «Недостаточный». Значения, которые не подходят ни под один шаблон, помечаются как `РЕЗУЛЬТАТ_ЗНАЧЕНИЕ`.

In [ ]:
print('=== Правило 1: Нормализация result ===')
print('Уникальные значения до:', df['result'].value_counts().to_dict())

fixed_result = 0
for idx, row in df.iterrows():
    r = row['result'].strip()
    r_up = r.upper()
    if r_up in ('ДОСТАТОЧНЫЙ', 'достаточный'.upper()):
        # уже правильно
        continue
    if 'ДОСТАТОЧН' in r_up:
        if 'НЕ' in r_up or r_up.startswith('НЕД'):
            normalized = 'Недостаточный'
        else:
            normalized = 'Достаточный'
        if r != normalized:
            fix_log(idx, 'result', r, normalized, f'Исправление регистра: "{r}" → "{normalized}"')
            df.at[idx, 'result'] = normalized
            fixed_result += 1
    else:
        flag(idx, 'РЕЗУЛЬТАТ_ЗНАЧЕНИЕ', f'Неизвестное значение result: "{r}"')

print(f'\nИсправлено: {fixed_result} записей')
print('Уникальные значения после:', df['result'].value_counts().to_dict())

---
## Правило 2. Нормализация поля `id_doc` (документ ребёнка)

**Проблема:** номер документа ребёнка иногда содержит:
- Буквы перед цифрами: `-N0849879` — буква N является артефактом экспорта
- Знак минус: `-1236233` — отрицательный номер физически невозможен
- Нечисловые символы: пробелы, точки

**Стратегия: FIX (исправление).**
1. Убираем все не-цифровые символы (кроме ведущего минуса), получаем числовую строку.
2. Если после очистки значение отрицательное — берём абсолютное значение.
3. Если после очистки значение пустое — помечаем как `ДОКУМЕНТ_РЕБЁНОК`.

In [ ]:
print('=== Правило 2: Нормализация id_doc ===')

fixed_doc = 0
invalid_doc = 0
for idx, row in df.iterrows():
    orig = row['id_doc'].strip()
    if not orig:
        flag(idx, 'ДОКУМЕНТ_РЕБЁНОК', 'Пустое поле id_doc')
        invalid_doc += 1
        continue

    # Шаг 1: убираем все буквы и спецсимволы кроме цифр и знака минус в начале
    cleaned = re.sub(r'[^0-9-]', '', orig)  # оставим только цифры и дефисы
    cleaned = re.sub(r'(?<!^)-', '', cleaned)  # убираем все дефисы кроме ведущего
    
    # Шаг 2: если отрицательное — берём abs
    try:
        val = int(cleaned)
        final = str(abs(val))
    except (ValueError, OverflowError):
        flag(idx, 'ДОКУМЕНТ_РЕБЁНОК', f'Нечисловой id_doc после очистки: "{orig}"')
        invalid_doc += 1
        continue

    if final != orig:
        fix_log(idx, 'id_doc', orig, final, 'Очистка нечисловых символов / убран знак минус')
        df.at[idx, 'id_doc'] = final
        fixed_doc += 1

print(f'Исправлено: {fixed_doc}')
print(f'Невалидных (помечено): {invalid_doc}')

---
## Правило 3. Нормализация поля `guard_id_doc` (документ представителя)

**Проблема:** аналогична `id_doc` — встречаются отрицательные значения и буквенные артефакты.

**Стратегия: FIX (исправление)** — та же, что и для `id_doc`.

In [ ]:
print('=== Правило 3: Нормализация guard_id_doc ===')

fixed_gdoc = 0
invalid_gdoc = 0
for idx, row in df.iterrows():
    orig = row['guard_id_doc'].strip()
    if not orig:
        flag(idx, 'ДОКУМЕНТ_ПРЕДСТАВИТЕЛЬ', 'Пустое поле guard_id_doc')
        invalid_gdoc += 1
        continue

    cleaned = re.sub(r'[^0-9-]', '', orig)
    cleaned = re.sub(r'(?<!^)-', '', cleaned)
    
    try:
        val = int(cleaned)
        final = str(abs(val))
    except (ValueError, OverflowError):
        flag(idx, 'ДОКУМЕНТ_ПРЕДСТАВИТЕЛЬ', f'Нечисловой guard_id_doc после очистки: "{orig}"')
        invalid_gdoc += 1
        continue

    if final != orig:
        fix_log(idx, 'guard_id_doc', orig, final, 'Очистка нечисловых символов / убран знак минус')
        df.at[idx, 'guard_id_doc'] = final
        fixed_gdoc += 1

print(f'Исправлено: {fixed_gdoc}')
print(f'Невалидных (помечено): {invalid_gdoc}')

---
## Правило 4. Совпадение номеров документов ребёнка и представителя

**Проблема:** `id_doc == guard_id_doc` — один и тот же номер документа не может принадлежать одновременно ребёнку и его законному представителю. Это либо ошибка ввода, либо намеренная подмена данных.

**Стратегия: FLAG (аномалия, не исправляется)** — исправить автоматически невозможно, т.к. неизвестно, какое из двух значений верное. Требует ручной проверки.

In [ ]:
print('=== Правило 4: id_doc == guard_id_doc ===')

same_doc = 0
for idx, row in df.iterrows():
    doc  = row['id_doc'].strip()
    gdoc = row['guard_id_doc'].strip()
    if doc and gdoc and doc == gdoc:
        flag(idx, 'ДОКУМЕНТ_СОВПАДЕНИЕ',
             f'id_doc == guard_id_doc = {doc} (один номер на ребёнка и представителя)')
        same_doc += 1

print(f'Совпадений: {same_doc}')

---
## Правило 5. Пустые обязательные поля

**Проблема:** ряд полей является обязательным. Пустое значение в любом из них означает неполноту записи и невозможность однозначной идентификации ребёнка или события тестирования.

Обязательные поля: `last_name`, `first_name`, `bdate`, `gender`, `id_doc`, `guard_last_name`, `guard_first_name`, `guard_bdate`, `our_number`, `ogrn_naprav`, `name_naprav`, `ogrn_area`, `name_area`, `variant`, `class`, `test_date`, `result`.

**Стратегия: FLAG** — пустое обязательное поле не поддаётся автоматическому исправлению.

In [ ]:
print('=== Правило 5: Пустые обязательные поля ===')

REQUIRED = [
    'last_name', 'first_name', 'bdate', 'gender', 'id_doc',
    'guard_last_name', 'guard_first_name', 'guard_bdate',
    'our_number', 'ogrn_naprav', 'name_naprav',
    'ogrn_area', 'name_area', 'variant', 'class', 'test_date', 'result',
]

empty_count = 0
field_stats = Counter()
for idx, row in df.iterrows():
    for field in REQUIRED:
        if not row[field].strip():
            flag(idx, 'ПУСТОЕ_ПОЛЕ', f'Пустое обязательное поле: "{field}"')
            field_stats[field] += 1
            empty_count += 1

print(f'Всего пустых обязательных полей: {empty_count}')
print('По полям:')
for f, c in field_stats.most_common():
    print(f'  {f}: {c}')

---
## Правило 6. Нормализация ОГРН направившей организации и площадки

**Проблема:** ОГРН (основной государственный регистрационный номер) состоит ровно из 13 цифр. Встречаются:
- ОГРН с пробелами: `«102 240 248 730 4»` → легко исправить, убрав пробелы
- ОГРН неверной длины — это уже реальная ошибка

**Стратегия: FIX + FLAG.** Убираем пробелы и незначимые символы; если после этого не 13 цифр — помечаем как аномалию.

In [ ]:
print('=== Правило 6: ОГРН направившей организации и площадки ===')

fixed_ogrn = 0
bad_ogrn = 0
for idx, row in df.iterrows():
    for field, cat in [('ogrn_naprav', 'ОГРН_НАПРАВИВШАЯ'), ('ogrn_area', 'ОГРН_ПЛОЩАДКА')]:
        orig = row[field].strip()
        # Убираем пробелы и нецифровые символы
        cleaned = re.sub(r'\D', '', orig)
        if cleaned != orig.replace(' ', ''):
            # Были нецифровые символы — попробуем исправить
            pass
        if re.match(r'^\d{13}$', cleaned):
            if cleaned != orig:
                fix_log(idx, field, orig, cleaned, 'Убраны нецифровые символы из ОГРН')
                df.at[idx, field] = cleaned
                fixed_ogrn += 1
        else:
            flag(idx, cat, f'Некорректный ОГРН ({field}): "{orig}" (длина {len(cleaned)} вместо 13)')
            bad_ogrn += 1

print(f'ОГРН исправлено: {fixed_ogrn}')
print(f'Невалидных ОГРН: {bad_ogrn}')

---
## Правило 7. Формат и диапазон поля `class` (класс обучения)

**Проблема:** поле `class` должно быть целым числом от 1 до 11. В датасете обнаружен случай значения `«2-5»` (диапазон классов) — это не может быть автоматически исправлено. Также проверяем выход из диапазона.

**Стратегия: FLAG** — нечисловые значения и значения вне диапазона 1–11 помечаются как аномалия.

In [ ]:
print('=== Правило 7: Формат и диапазон класса ===')

bad_class = 0
for idx, row in df.iterrows():
    cls_str = row['class'].strip()
    try:
        cls = int(cls_str)
        if not (1 <= cls <= 11):
            flag(idx, 'КЛАСС_ДИАПАЗОН', f'Класс вне диапазона 1–11: {cls}')
            bad_class += 1
    except ValueError:
        flag(idx, 'КЛАСС_ФОРМАТ', f'Нечисловое значение класса: "{cls_str}"')
        bad_class += 1

print(f'Записей с некорректным классом: {bad_class}')

---
## Правило 8. Формат и содержание кода варианта (`variant`)

**Проблема:** код варианта тестирования должен быть строкой из 5–7 цифр. Встречаются:
- Слишком короткие значения: `«0»`, `«1»`, `«10»`
- Дробные числа: `«727.040501»`
- Даты: `«08.09.2025»`
- Строки с пробелами и буквами

Дополнительно: средние две цифры кода кодируют класс (позиции `[-4:-2]`). Пример: `80109` → позиции 1–2 = `01` → класс 1.

**Стратегия: FLAG** — некорректный формат не исправляется автоматически.

In [ ]:
print('=== Правило 8: Формат варианта и соответствие классу ===')

bad_fmt = 0
bad_cls_match = 0
for idx, row in df.iterrows():
    v   = row['variant'].strip()
    cls = row['class'].strip()

    if not re.match(r'^\d{5,7}$', v):
        flag(idx, 'ВАРИАНТ_ФОРМАТ', f'Некорректный формат варианта: "{v}" (ожидается 5–7 цифр)')
        bad_fmt += 1
        continue  # дальше не проверяем

    # Проверяем встроенный класс (позиции v[-4:-2])
    try:
        embedded = int(v[-4:-2])
        actual   = int(cls)
        if embedded != actual:
            flag(idx, 'ВАРИАНТ_КЛАСС',
                 f'Вариант "{v}" кодирует класс {embedded}, но в поле class = {actual}')
            bad_cls_match += 1
    except (ValueError, IndexError):
        pass

print(f'Некорректный формат варианта: {bad_fmt}')
print(f'Несоответствие варианта и класса: {bad_cls_match}')

---
## Правило 9. Дата рождения ребёнка: диапазон и корректность

**Проблема:** дата рождения не может быть:
- **В будущем** относительно даты анализа
- **Позже даты тестирования** (ребёнок ещё не родился, а уже тестируется)
- Такой, что **возраст на дату теста < 5 лет** (слишком мал для тестирования)
- Такой, что **возраст > 20 лет** (взрослый, явно не школьник)

Дополнительно: по закону РФ, в 1-й класс принимают детей, которым исполнилось **не менее 6 лет 6 месяцев** к 1 сентября.

**Стратегия: FLAG** — даты не исправляются автоматически.

In [ ]:
print('=== Правило 9: Дата рождения ребёнка ===')

bad_bdate = 0
too_young = 0
for idx, row in df.iterrows():
    try:
        bdate     = datetime.strptime(row['bdate'], '%Y-%m-%d').date()
        test_date = datetime.strptime(row['test_date'], '%Y-%m-%d').date()
    except ValueError:
        flag(idx, 'ДАТА_РОЖДЕНИЯ', f'Некорректный формат даты: bdate="{row["bdate"]}" test_date="{row["test_date"]}"')
        bad_bdate += 1
        continue

    age = (test_date - bdate).days / 365.25

    if bdate > TODAY:
        flag(idx, 'ДАТА_РОЖДЕНИЯ', f'Дата рождения в будущем: {bdate}')
        bad_bdate += 1
    elif bdate > test_date:
        flag(idx, 'ДАТА_РОЖДЕНИЯ', f'bdate ({bdate}) позже test_date ({test_date})')
        bad_bdate += 1
    elif age < 5:
        flag(idx, 'ДАТА_РОЖДЕНИЯ', f'Возраст на тест: {age:.1f} лет (< 5)')
        bad_bdate += 1
    elif age > 20:
        flag(idx, 'ДАТА_РОЖДЕНИЯ', f'Возраст на тест: {age:.1f} лет (> 20)')
        bad_bdate += 1
    else:
        # Доп. проверка: возраст на 1 сентября года тестирования < 6.5
        sep1 = date(test_date.year, 9, 1)
        age_sep = (sep1 - bdate).days / 365.25
        if age_sep < 6.5:
            flag(idx, 'ДАТА_РОЖДЕНИЯ', f'Возраст на 1 сентября {test_date.year}: {age_sep:.1f} лет (< 6.5, по РФ не принимают в школу)')
            too_young += 1

print(f'Аномальных дат рождения: {bad_bdate}')
print(f'Слишком маленький возраст на 1 сентября: {too_young}')

---
## Правило 10. Соответствие возраста ребёнка и класса обучения

**Проблема:** в российской школьной системе ребёнок в классе N должен иметь возраст в диапазоне **N+5 … N+8 лет** на дату тестирования. Например:
- Класс 1 → возраст 6–9 лет
- Класс 5 → возраст 10–13 лет
- Класс 11 → возраст 16–19 лет

Выход за этот диапазон означает либо ошибку в дате рождения, либо ошибку в классе.

**Стратегия: FLAG** — нельзя автоматически определить, что именно неверно.

In [ ]:
print('=== Правило 10: Соответствие возраста и класса ===')

age_cls_issues = 0
for idx, row in df.iterrows():
    try:
        bdate     = datetime.strptime(row['bdate'], '%Y-%m-%d').date()
        test_date = datetime.strptime(row['test_date'], '%Y-%m-%d').date()
        cls       = int(row['class'])
    except ValueError:
        continue  # дату/класс уже проверили

    age = (test_date - bdate).days / 365.25
    age_min = cls + 5
    age_max = cls + 8

    if not (age_min <= age <= age_max):
        flag(idx, 'ВОЗРАСТ_КЛАСС',
             f'Возраст {age:.1f} лет не соответствует классу {cls} (ожидается {age_min}–{age_max})')
        age_cls_issues += 1

print(f'Несоответствий возраста и класса: {age_cls_issues}')

---
## Правило 11. Возраст законного представителя

**Проблема:** по российскому законодательству, минимальный возраст родителя — 14 лет (несовершеннолетние могут быть признаны дееспособными). На практике законный представитель должен быть **не менее чем на 14 лет старше** ребёнка. Если разница меньше — скорее всего ошибка в дате рождения представителя.

**Стратегия: FLAG** — нельзя определить, чья дата рождения неверна.

In [ ]:
print('=== Правило 11: Возраст законного представителя ===')

young_guard = 0
for idx, row in df.iterrows():
    try:
        child_bdate = datetime.strptime(row['bdate'], '%Y-%m-%d').date()
        guard_bdate = datetime.strptime(row['guard_bdate'], '%Y-%m-%d').date()
    except ValueError:
        continue

    age_diff = (child_bdate - guard_bdate).days / 365.25
    if age_diff < 14:
        flag(idx, 'ПРЕДСТАВИТЕЛЬ_ВОЗРАСТ',
             f'Представитель {row["guard_last_name"]} старше ребёнка '
             f'только на {age_diff:.1f} лет (норма ≥ 14)')
        young_guard += 1

print(f'Слишком молодых представителей: {young_guard}')

---
## Правило 12. Нарушение частоты тестирования

**Проблема:** согласно заданию ЦИТиС, тестирование не должно проводиться чаще **1 раза в 3 месяца (90 дней)** для одного и того же ребёнка. Идентификатор ребёнка — комбинация фамилии, имени и даты рождения (не `id_doc`, т.к. он может содержать ошибки).

Для каждого ребёнка сортируем его тесты по дате и проверяем, нет ли пары подряд идущих тестов с интервалом < 90 дней.

**Стратегия: FLAG** — нарушение требует ручного анализа (какой из тестов «лишний»).

In [ ]:
print('=== Правило 12: Частота тестирования (≥ 90 дней между тестами) ===')

child_tests = defaultdict(list)
for idx, row in df.iterrows():
    try:
        td = datetime.strptime(row['test_date'], '%Y-%m-%d').date()
        key = (row['last_name'].upper(), row['first_name'].upper(), row['bdate'])
        child_tests[key].append((td, idx))
    except ValueError:
        pass

freq_violations = 0
for key, tests in child_tests.items():
    tests_sorted = sorted(tests, key=lambda x: x[0])
    for j in range(1, len(tests_sorted)):
        prev_date, prev_idx = tests_sorted[j - 1]
        curr_date, curr_idx = tests_sorted[j]
        diff = (curr_date - prev_date).days
        if diff < 90:
            flag(curr_idx, 'ЧАСТОТА',
                 f'Тест через {diff} дн. после предыдущего ({prev_date} → {curr_date}), норма ≥ 90')
            freq_violations += 1

print(f'Нарушений частоты: {freq_violations}')

---
## Правило 13. Соответствие имени и пола ребёнка

**Проблема:** поле `gender` должно соответствовать имени ребёнка. Несоответствие возникает из-за ошибки ввода. Используем морфологический анализатор **pymorphy2** для определения грамматического рода имени.

**Стратегия: FLAG** — нельзя автоматически определить, что именно неверно (пол или имя).

In [ ]:
print('=== Правило 13: Соответствие имени и пола ===')

FEMALE_EXCEPTIONS = {
    'антонина','евгения','бронислава','серафима','валерия','эльмира',
    'анжела','федосия','алена','асия','альфия','павла','варвара',
    'владислава','яна','алефтина','ольга','елена','камия','таисия',
    'александра','римма',
}
MALE_EXCEPTIONS = {'михаил','ивлиане','илтизир'}

try:
    import pymorphy2
    morph = pymorphy2.MorphAnalyzer()
    gender_errors = 0

    for idx, row in df.iterrows():
        gender_val = row['gender'].strip()
        name = row['first_name'].strip()
        if not gender_val or not name:
            continue

        expected = 'femn' if gender_val == 'Ж' else 'masc'
        name_lower = name.lower()

        if name_lower in FEMALE_EXCEPTIONS:
            actual = 'femn'
        elif name_lower in MALE_EXCEPTIONS:
            actual = 'masc'
        else:
            parsed = morph.parse(name)[0]
            grms = parsed.tag.grammemes
            actual = 'femn' if 'femn' in grms else ('masc' if 'masc' in grms else 'unknown')

        if actual != 'unknown' and actual != expected:
            expected_str = 'женское' if expected == 'femn' else 'мужское'
            actual_str   = 'женское' if actual   == 'femn' else 'мужское'
            flag(idx, 'ПОЛ_ИМЯ', f'Имя "{name}" ({actual_str}), но gender={gender_val} ({expected_str})')
            gender_errors += 1

    print(f'Несоответствий пола и имени: {gender_errors}')

except Exception as e:
    print(f'Правило пропущено: {e}')
    print('Установите совместимую версию: pip install pymorphy2 pymorphy2-dicts-ru')

---
## Итоговая статистика ручных проверок

Подводим итоги: сколько записей исправлено, сколько помечено как аномалии, как они распределены по категориям.

In [ ]:
print('=' * 60)
print('ИТОГИ РУЧНОЙ ПРОВЕРКИ')
print('=' * 60)

n_total    = len(df)
n_fixed    = (df['fixes_applied'] != '').sum()
n_flagged  = (df['anomaly_flags'] != '').sum()
n_clean    = n_total - n_flagged

print(f'Всего записей:            {n_total}')
print(f'Исправлено (FIX):         {n_fixed}  ({n_fixed/n_total*100:.1f}%)')
print(f'Помечено как аномалия:    {n_flagged}  ({n_flagged/n_total*100:.1f}%)')
print(f'Чистых записей:           {n_clean}  ({n_clean/n_total*100:.1f}%)')

# Статистика по категориям
cat_counts = Counter(a['category'] for a in anomalies)
print('\nПо категориям:')
for cat, cnt in sorted(cat_counts.items(), key=lambda x: -x[1]):
    print(f'  {cat:<30} {cnt:>6}')

# Сохраняем отчёты
pd.DataFrame(anomalies).to_csv(f'{OUT_DIR}/manual_anomalies.csv', index=False, encoding='utf-8')
pd.DataFrame(fixes_log).to_csv(f'{OUT_DIR}/fixes_log.csv', index=False, encoding='utf-8')
print(f'\nОтчёты сохранены в {OUT_DIR}/')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['DejaVu Sans']

cats = [(c, n) for c, n in sorted(cat_counts.items(), key=lambda x: -x[1])]
labels = [c for c, _ in cats]
values = [n for _, n in cats]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart — категории аномалий
axes[0].barh(labels[::-1], values[::-1], color='#E63946')
axes[0].set_title('Количество аномалий по категориям')
axes[0].set_xlabel('Количество записей')
for i, v in enumerate(values[::-1]):
    axes[0].text(v + 10, i, str(v), va='center', fontsize=9)

# Pie — доли в датасете
slices = [n_flagged, n_fixed - (df['anomaly_flags'] != '').sum() + (df['anomaly_flags'] != '').sum(), n_clean]
axes[1].pie(
    [n_flagged, n_clean],
    labels=[f'Аномалии ({n_flagged})', f'Чистые ({n_clean})'],
    colors=['#E63946', '#457B9D'],
    autopct='%1.1f%%', startangle=90,
)
axes[1].set_title('Распределение записей после ручной проверки')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/manual_stats.png', dpi=120, bbox_inches='tight')
plt.show()
print('График сохранён.')

---
## Подготовка чистого датасета для Isolation Forest

Записи, помеченные как аномалии ручными правилами, исключаются из обучающей выборки. Остальные записи — включая исправленные — используются для обучения модели.

Также создаём числовые признаки для Isolation Forest:
- `age_at_test` — возраст ребёнка на дату теста (лет)
- `class_num` — класс обучения
- `age_class_dev` — отклонение возраста от ожидаемого среднего для класса
- `guard_age_diff` — разница в возрасте между представителем и ребёнком
- `result_code` — 1 = «Достаточный», 0 = «Недостаточный»
- `gender_code` — 1 = М, 0 = Ж
- `test_month` — месяц тестирования (сезонность)
- `test_year` — год тестирования
- `tests_per_child` — сколько раз ребёнок тестировался всего
- `days_since_prev` — дней с предыдущего теста (−1 если первый тест)

In [ ]:
print('=== Подготовка чистого датасета ===')

# Отбираем только чистые записи (без флагов аномалий)
df_clean = df[df['anomaly_flags'] == ''].copy().reset_index(drop=True)
print(f'Чистых записей для обучения: {len(df_clean)} из {len(df)}')

# ──────────────────── Числовые признаки ────────────────────
df_clean['bdate_dt']      = pd.to_datetime(df_clean['bdate'], errors='coerce')
df_clean['test_date_dt']  = pd.to_datetime(df_clean['test_date'], errors='coerce')
df_clean['guard_bdate_dt']= pd.to_datetime(df_clean['guard_bdate'], errors='coerce')
df_clean['class_num']     = pd.to_numeric(df_clean['class'], errors='coerce')

df_clean['age_at_test']   = ((df_clean['test_date_dt'] - df_clean['bdate_dt']).dt.days / 365.25)
df_clean['age_class_dev'] = df_clean['age_at_test'] - (df_clean['class_num'] + 6.5)
df_clean['guard_age_diff']= ((df_clean['bdate_dt'] - df_clean['guard_bdate_dt']).dt.days / 365.25)
df_clean['result_code']   = (df_clean['result'].str.upper().str.strip().str.contains('НЕДОСТ') == False).astype(int)
df_clean['gender_code']   = (df_clean['gender'].str.strip() == 'М').astype(int)
df_clean['test_month']    = df_clean['test_date_dt'].dt.month
df_clean['test_year']     = df_clean['test_date_dt'].dt.year

# Частота тестирований на ребёнка
key_col = df_clean['last_name'].str.upper() + '|' + df_clean['first_name'].str.upper() + '|' + df_clean['bdate']
df_clean['tests_per_child'] = key_col.map(key_col.value_counts())

# Дней с предыдущего теста
df_clean = df_clean.sort_values(['last_name', 'first_name', 'bdate', 'test_date_dt'])
df_clean['days_since_prev'] = df_clean.groupby([df_clean['last_name'].str.upper(),
                                                  df_clean['first_name'].str.upper(),
                                                  'bdate'])['test_date_dt'].diff().dt.days.fillna(-1)
df_clean = df_clean.reset_index(drop=True)

FEATURES = ['age_at_test', 'class_num', 'age_class_dev', 'guard_age_diff',
            'result_code', 'gender_code', 'test_month', 'test_year',
            'tests_per_child', 'days_since_prev']

# Удаляем строки с NaN в признаках
df_model = df_clean.dropna(subset=FEATURES).copy()
print(f'Записей с полными признаками: {len(df_model)}')

# Сохраняем чистый датасет
df_model.to_csv(f'{OUT_DIR}/clean_dataset.csv', sep=';', index=False, encoding='utf-8')
print(f'Чистый датасет сохранён: {OUT_DIR}/clean_dataset.csv')

---
## Обучение Isolation Forest на чистых данных

**Логика алгоритма:** Isolation Forest строит множество случайных деревьев. Каждое дерево последовательно разбивает пространство признаков случайными границами. Для «нормальной» точки требуется много разрезов, чтобы её изолировать — она находится в плотном облаке. Для аномальной — мало разрезов, т.к. она «одиночка».

**Идея:** модель обучается на «чистых» данных (после исключения всех ручных аномалий). Записи, которые модель с трудом помещает в выученное распределение (т.е. изолирует слишком быстро), — это **новые, ранее неизвестные аномалии**.

**Параметры:**
- `n_estimators=200` — количество деревьев (больше → стабильнее)
- `contamination=0.05` — ожидаемая доля аномалий (5%)
- `max_samples='auto'` — 256 образцов на дерево

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

print('=== Обучение Isolation Forest ===')

X = df_model[FEATURES].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iforest = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    max_samples='auto',
    n_jobs=-1,
    random_state=42,
)
iforest.fit(X_scaled)

# -1 = аномалия, 1 = норма (sklearn convention)
preds  = iforest.predict(X_scaled)
scores = iforest.decision_function(X_scaled)  # чем ниже — тем аномальнее

df_model['iforest_pred']  = preds
df_model['anomaly_score'] = scores
df_model['is_ml_anomaly'] = (preds == -1).astype(int)

n_ml_anomalies = (preds == -1).sum()
print(f'Всего обучено на: {len(df_model)} записях')
print(f'ML-аномалий найдено: {n_ml_anomalies} ({n_ml_anomalies/len(df_model)*100:.1f}%)')
print(f'Диапазон anomaly_score: [{scores.min():.3f}, {scores.max():.3f}]')

---
## Анализ ML-аномалий: что нашёл Isolation Forest?

Смотрим на характеристики записей, помеченных моделью. Это позволяет понять, какие **паттерны** модель считает «нетипичными» для обучающей выборки.

In [ ]:
df_ml_anom = df_model[df_model['is_ml_anomaly'] == 1].copy()
df_ml_norm = df_model[df_model['is_ml_anomaly'] == 0].copy()

print(f'ML аномалии: {len(df_ml_anom)}')
print(f'ML норма:    {len(df_ml_norm)}')

print('\n--- Сравнение средних значений признаков ---')
comparison = pd.DataFrame({
    'Норма':    df_ml_norm[FEATURES].mean(),
    'Аномалия': df_ml_anom[FEATURES].mean(),
})
comparison['Разница'] = comparison['Аномалия'] - comparison['Норма']
display(comparison.round(2))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

plot_pairs = [
    ('anomaly_score', 'Распределение anomaly_score', 'Счёт (ниже = аномальнее)'),
    ('age_at_test',   'Возраст на тест',              'Лет'),
    ('class_num',     'Класс обучения',               'Класс'),
    ('guard_age_diff','Разница возраста (представитель–ребёнок)', 'Лет'),
    ('tests_per_child','Тестов на ребёнка',           'Количество тестов'),
    ('test_month',    'Месяц тестирования',           'Месяц'),
]

for ax, (col, title, xlabel) in zip(axes.flat, plot_pairs):
    ax.hist(df_ml_norm[col],  bins=30, alpha=0.6, color='#457B9D', label='Норма')
    ax.hist(df_ml_anom[col],  bins=30, alpha=0.7, color='#E63946', label='ML аномалия')
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.legend(fontsize=8)

plt.suptitle('Сравнение ML-аномалий и нормы по признакам', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/ml_feature_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
print('--- Топ-20 ML-аномалий (наиболее аномальные) ---')
top_cols = ['our_number', 'last_name', 'first_name', 'bdate', 'class', 'test_date',
            'result', 'anomaly_score', 'age_at_test', 'tests_per_child']
display(
    df_ml_anom.sort_values('anomaly_score').head(20)[top_cols].reset_index(drop=True)
)

---
## Сохранение всех результатов и итоговая сводка

In [ ]:
# Сохраняем полный датасет с признаками и метками
df_model.to_csv(f'{OUT_DIR}/full_scored.csv', sep=';', index=False, encoding='utf-8')

# Только ML-аномалии
df_ml_anom.to_csv(f'{OUT_DIR}/ml_anomalies.csv', sep=';', index=False, encoding='utf-8')

# Сводная статистика
summary = {
    'total_records':        len(df),
    'fixed_records':        int(n_fixed),
    'manual_anomalies':     int(n_flagged),
    'clean_for_ml':         len(df_model),
    'ml_anomalies':         int(n_ml_anomalies),
    'truly_clean':          int(len(df_model) - n_ml_anomalies),
}

pd.DataFrame([summary]).to_csv(f'{OUT_DIR}/summary.csv', index=False, encoding='utf-8')

print('=' * 55)
print('ФИНАЛЬНАЯ СВОДКА')
print('=' * 55)
print(f"Всего записей в датасете:        {summary['total_records']}")
print(f"Исправлено (FIX-правила):        {summary['fixed_records']}")
print(f"Ручных аномалий (FLAG):          {summary['manual_anomalies']}")
print(f"Подано на Isolation Forest:      {summary['clean_for_ml']}")
print(f"Новых ML-аномалий:               {summary['ml_anomalies']}")
print(f"Полностью чистых записей:        {summary['truly_clean']}")
print()
print('Файлы:')
for fn in os.listdir(OUT_DIR):
    size = os.path.getsize(f'{OUT_DIR}/{fn}')
    print(f'  {fn:<35} {size//1024:>5} KB')

---
## Визуализация воронки обработки данных

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

stages = [
    ('Исходный датасет',     summary['total_records'],   '#264653'),
    ('Исправлено (FIX)',     summary['fixed_records'],   '#2a9d8f'),
    ('Ручных аномалий',      summary['manual_anomalies'],'#e9c46a'),
    ('Подано в IF',          summary['clean_for_ml'],    '#457B9D'),
    ('ML-аномалий',          summary['ml_anomalies'],    '#E63946'),
    ('Чистых итого',         summary['truly_clean'],     '#A8DADC'),
]

labels = [s[0] for s in stages]
values = [s[1] for s in stages]
colors = [s[2] for s in stages]

bars = ax.barh(range(len(stages)), values, color=colors, edgecolor='white', height=0.6)
ax.set_yticks(range(len(stages)))
ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel('Количество записей')
ax.set_title('Воронка обработки данных', fontsize=14)

for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/processing_funnel.png', dpi=120, bbox_inches='tight')
plt.show()